In [1]:
import pandas as pd 
from torchmetrics.text.rouge import ROUGEScore
from transformers import AutoTokenizer, AutoModelForCausalLM
import re

/work/hpc/miniconda3/envs/hieu-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/work/hpc/miniconda3/envs/hieu-env/lib/python3.12/site-packages/transformers/utils/hub.py:127: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [4]:
data_path = "logger/gemma-2-2b-it+domain_masked_wrong_answer/result.csv"
model_name = "google/gemma-2-2b-it"

In [5]:
df = pd.read_csv(data_path)

In [6]:
df.head()

,Unnamed: 0.4,Unnamed: 0.3,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,Question,Answer,Label,rouge_score,check_score
0,0,0,0,0,0,<bos><start_of_turn>user\nHãy điền vào trong d...,.\nmodel\nA: Báo với cô giáo chủ nhiệm để tìm ...,Chạy đi chỗ khác chơi.,0.357143,0.000000
1,1,1,1,1,1,<bos><start_of_turn>user\nHãy điền vào trong d...,.\nmodel\nA: **Đưa em bé đó đến bệnh viện và g...,Lờ đi chỗ khác và coi như không biết.,0.121212,0.071429
2,2,2,2,2,2,<bos><start_of_turn>user\nHãy điền vào trong d...,.\nmodel\nA: **Gió chiều nào theo chiều ấy nói...,Không tôn trọng lẽ phải.,0.206897,0.000000
3,3,3,3,3,3,<bos><start_of_turn>user\nHãy điền vào trong d...,".\nmodel\nC: **Làm giàu bằng chính sức lực, tà...",Nhặt được của rơi trả lại cho người bị mất,0.315789,0.086957
4,4,4,4,4,4,<bos><start_of_turn>user\nHãy điền vào trong d...,.\nmodel\nB: Liêm khiết. \n \n,Công bằng.,0.200000,0.000000


In [7]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

In [8]:
def normalizer(text):
    text = re.sub(r"[^a-z0-9àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ]+", " ", text.lower())
    text = text.strip(" ")
    return text

rouge = ROUGEScore(rouge_keys="rougeL", tokenizer=tokenizer.tokenize, use_stemmer= False, normalizer=normalizer)

In [10]:
res = []
for _, row in df.iterrows():
    target = row["Label"]
    prediction = row["Answer"]
    score = rouge(prediction, target)
    res.append(score)
    break

In [11]:
res

[{'rougeL_fmeasure': tensor(0.),
  'rougeL_precision': tensor(0.),
  'rougeL_recall': tensor(0.)}]

In [12]:
df["check_score"] = res

In [17]:
df["check_score"] = df["check_score"].apply(lambda x: x["rougeL_fmeasure"].item())

In [18]:
df["check_score"].mean()

0.20326764406048478

In [19]:
df["rouge_score"].mean()

0.30902643013565406